# SurfaceTarget-CRC — phase 01 — patch 1 19

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# PHASE 1 — PATCH 1.19
#
# RNA-SEQ PCA AND TECHNICAL CONFOUNDING ASSESSMENT
#
# Objectives:
# 1. Load the validated expression matrix
# 2. Select informative protein-coding genes
# 3. Calculate exploratory log2(CPM + 1) expression
# 4. Perform whole-cohort PCA
# 5. Examine workflow-associated expression structure
# 6. Perform workflow-restricted COAD and READ PCA
# 7. Highlight samples previously flagged for QC review
# 8. Save PCA coordinates, genes, figures and provenance
#
# IMPORTANT:
# - No samples are excluded.
# - No batch correction is applied.
# - No differential expression is performed.
# - PCA is exploratory, not a causal analysis.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

from sklearn.decomposition import PCA


# ------------------------------------------------------------
# 1. Project configuration
# ------------------------------------------------------------

ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
PROCESSED = ROOT / "data/processed"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"

RESULTS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

MATRIX_PATH = (
    PROCESSED / "tcga_unstranded_raw_counts.npz"
)

GENES_PATH = META / "tcga_gene_annotations.csv"

SAMPLES_PATH = (
    META / "tcga_expression_sample_metadata.csv"
)

QC_PATH = RESULTS / "tcga_full_cohort_qc.csv"

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.19 — EXPRESSION PCA AND CONFOUNDING")
print("=" * 65)


# ------------------------------------------------------------
# 2. Load expression data
# ------------------------------------------------------------

for path in [
    MATRIX_PATH,
    GENES_PATH,
    SAMPLES_PATH,
    QC_PATH
]:
    if not path.exists():
        raise FileNotFoundError(path)

with np.load(
    MATRIX_PATH,
    allow_pickle=False
) as data:

    counts = data["counts"]
    gene_ids = data["gene_ids"]
    file_ids = data["file_ids"]

genes = pd.read_csv(
    GENES_PATH,
    dtype=str,
    keep_default_na=False
)

samples = pd.read_csv(
    SAMPLES_PATH,
    dtype=str,
    keep_default_na=False
)

qc = pd.read_csv(
    QC_PATH,
    dtype=str,
    keep_default_na=False
)

print("\nData loaded successfully.")

print("Expression matrix:", counts.shape)
print("Sample records:", len(samples))
print("QC records:", len(qc))


# ------------------------------------------------------------
# 3. Validate sample alignment
# ------------------------------------------------------------

if counts.shape != (60660, 472):
    raise RuntimeError(
        "Unexpected expression matrix dimensions."
    )

if not np.array_equal(
    file_ids,
    samples["file_id"].to_numpy()
):
    raise RuntimeError(
        "Matrix and sample metadata are misaligned."
    )

if not np.array_equal(
    file_ids,
    qc["file_id"].to_numpy()
):
    raise RuntimeError(
        "Matrix and QC records are misaligned."
    )

if not np.array_equal(
    gene_ids,
    genes["gene_id"].to_numpy()
):
    raise RuntimeError(
        "Gene identifiers are misaligned."
    )

if not np.issubdtype(
    counts.dtype,
    np.integer
):
    raise RuntimeError(
        "Raw count matrix must contain integers."
    )

qc["n_qc_review_flags"] = pd.to_numeric(
    qc["n_qc_review_flags"],
    errors="raise"
).astype(int)

qc["library_size_millions"] = pd.to_numeric(
    qc["library_size_millions"],
    errors="raise"
)

qc["assignment_percent"] = pd.to_numeric(
    qc["assignment_percent"],
    errors="raise"
)

print("\nSample and gene alignment: PASSED")


# ------------------------------------------------------------
# 4. Identify protein-coding genes
# ------------------------------------------------------------

coding_mask = (
    genes["gene_type"]
    .eq("protein_coding")
    .to_numpy(dtype=bool)
)

coding_counts = counts[
    coding_mask, :
]

coding_gene_ids = gene_ids[coding_mask]

print("\nProtein-coding genes:", len(coding_gene_ids))

if len(coding_gene_ids) != 19962:
    raise RuntimeError(
        "Unexpected protein-coding gene count."
    )

library_sizes = counts.sum(
    axis=0,
    dtype=np.int64
)

if np.any(library_sizes <= 0):
    raise RuntimeError(
        "Zero library size detected."
    )


# ------------------------------------------------------------
# 5. Identify shared workflow version
# ------------------------------------------------------------

# All normal samples should use the same
# recorded STAR workflow version.

normal_versions = samples.loc[
    samples["dataset_group"] == "Normal",
    "workflow_version"
].unique()

if len(normal_versions) != 1:
    raise RuntimeError(
        "Normal samples do not share one workflow."
    )

SHARED_WORKFLOW = normal_versions[0]

print("\nShared normal-tissue workflow:")
print(SHARED_WORKFLOW)

print("\nWorkflow-restricted sample availability:")

shared_samples = samples[
    samples["workflow_version"] == SHARED_WORKFLOW
]

print(
    pd.crosstab(
        shared_samples["project"],
        shared_samples["cohort_tier"]
    ).to_string()
)


# ------------------------------------------------------------
# 6. PCA analysis function
# ------------------------------------------------------------

def perform_pca(
    analysis_name,
    sample_indices,
    max_genes=2000
):

    sample_indices = np.asarray(
        sample_indices,
        dtype=int
    )

    subset = qc.iloc[
        sample_indices
    ].copy().reset_index(drop=True)

    n_samples = len(subset)

    if n_samples < 10:
        raise RuntimeError(
            f"Insufficient samples for {analysis_name}"
        )

    # --------------------------------------------
    # Extract protein-coding raw counts
    # --------------------------------------------

    subset_counts = coding_counts[
        :, sample_indices
    ]

    subset_libraries = library_sizes[
        sample_indices
    ]

    # --------------------------------------------
    # Calculate descriptive CPM
    # --------------------------------------------

    cpm = (
        subset_counts.astype(np.float64)
        / subset_libraries[np.newaxis, :]
        * 1_000_000
    )

    # --------------------------------------------
    # Filter minimally expressed genes
    #
    # CPM >= 1 in at least 10% of samples,
    # with a minimum of 10 samples.
    #
    # This filtering is ONLY for PCA.
    # --------------------------------------------

    min_detected_samples = max(
        10,
        int(np.ceil(n_samples * 0.10))
    )

    expressed_mask = (
        np.count_nonzero(
            cpm >= 1,
            axis=1
        )
        >= min_detected_samples
    )

    expressed_gene_indices = np.flatnonzero(
        expressed_mask
    )

    if len(expressed_gene_indices) < 100:
        raise RuntimeError(
            f"Too few expressed genes: {analysis_name}"
        )

    # --------------------------------------------
    # Log-transform expression
    # --------------------------------------------

    log_cpm = np.log2(
        cpm[expressed_mask, :] + 1.0
    )

    # --------------------------------------------
    # Select most variable genes
    # --------------------------------------------

    gene_variances = np.var(
        log_cpm,
        axis=1
    )

    ranked = np.argsort(
        -gene_variances,
        kind="stable"
    )

    n_selected = min(
        max_genes,
        int(np.count_nonzero(gene_variances > 0))
    )

    if n_selected < 100:
        raise RuntimeError(
            f"Too few variable genes: {analysis_name}"
        )

    top_indices = ranked[:n_selected]

    selected_gene_indices = (
        expressed_gene_indices[top_indices]
    )

    # Samples x genes.
    #
    # sklearn PCA automatically centers
    # each gene across samples.

    expression_for_pca = (
        log_cpm[top_indices, :].T
    )

    if not np.isfinite(
        expression_for_pca
    ).all():
        raise RuntimeError(
            "Nonfinite PCA expression values."
        )

    # --------------------------------------------
    # Fit PCA
    # --------------------------------------------

    model = PCA(
        n_components=3,
        svd_solver="randomized",
        random_state=42
    )

    coordinates = model.fit_transform(
        expression_for_pca
    )

    variance_explained = (
        model.explained_variance_ratio_ * 100
    )

    # --------------------------------------------
    # Save sample coordinates
    # --------------------------------------------

    for pc_index in range(3):

        subset[f"PC{pc_index + 1}"] = (
            coordinates[:, pc_index]
        )

    subset["pca_analysis"] = analysis_name

    # --------------------------------------------
    # Save genes used for PCA
    # --------------------------------------------

    selected_genes = pd.DataFrame({

        "gene_id":
            coding_gene_ids[selected_gene_indices],

        "log_cpm_variance":
            gene_variances[top_indices],

        "variance_rank":
            np.arange(1, n_selected + 1)
    })

    print("\n" + "-" * 60)
    print("PCA:", analysis_name)
    print("-" * 60)

    print("Samples:", n_samples)

    print(
        "Genes passing expression filter:",
        len(expressed_gene_indices)
    )

    print(
        "Variable genes selected:",
        n_selected
    )

    print(
        "Minimum samples for CPM filter:",
        min_detected_samples
    )

    print(
        "PC1 variance explained:",
        f"{variance_explained[0]:.2f}%"
    )

    print(
        "PC2 variance explained:",
        f"{variance_explained[1]:.2f}%"
    )

    print(
        "PC3 variance explained:",
        f"{variance_explained[2]:.2f}%"
    )

    print(
        "Previously flagged QC samples:",
        int(
            (subset["n_qc_review_flags"] > 0).sum()
        )
    )

    return (
        subset,
        selected_genes,
        variance_explained
    )


# ------------------------------------------------------------
# 7. PCA figure function
# ------------------------------------------------------------

def plot_pca(
    coordinates,
    variance,
    group_column,
    title,
    output_name
):

    fig, ax = plt.subplots(
        figsize=(10, 7)
    )

    groups = sorted(
        coordinates[group_column].unique()
    )

    for group in groups:

        group_data = coordinates[
            coordinates[group_column] == group
        ]

        ax.scatter(
            group_data["PC1"],
            group_data["PC2"],
            s=40,
            alpha=0.75,
            label=f"{group} (n={len(group_data)})"
        )

    # Highlight previously flagged QC samples
    # without removing them.

    flagged = coordinates[
        coordinates["n_qc_review_flags"] > 0
    ]

    if not flagged.empty:

        ax.scatter(
            flagged["PC1"],
            flagged["PC2"],
            marker="x",
            color="black",
            s=85,
            linewidths=1.3,
            label="QC review flag"
        )

    ax.set_xlabel(
        f"PC1 ({variance[0]:.1f}% variance)"
    )

    ax.set_ylabel(
        f"PC2 ({variance[1]:.1f}% variance)"
    )

    ax.set_title(title)

    ax.axhline(
        0,
        linewidth=0.5,
        alpha=0.35
    )

    ax.axvline(
        0,
        linewidth=0.5,
        alpha=0.35
    )

    ax.legend(
        fontsize=8,
        loc="best"
    )

    ax.grid(alpha=0.2)

    fig.tight_layout()

    output_path = FIGURES / output_name

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    print("Saved:", output_name)


# ------------------------------------------------------------
# 8. Whole-cohort PCA
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("WHOLE-COHORT EXPRESSION PCA")
print("=" * 65)

all_indices = np.arange(
    len(samples)
)

global_pca, global_genes, global_variance = (
    perform_pca(
        "global",
        all_indices
    )
)

plot_pca(
    global_pca,
    global_variance,
    "analysis_group",
    "TCGA RNA-seq PCA — Cohort Structure",
    "tcga_pca_global_by_cohort.png"
)

plot_pca(
    global_pca,
    global_variance,
    "workflow_short",
    "TCGA RNA-seq PCA — Workflow Versions",
    "tcga_pca_global_by_workflow.png"
)


# ------------------------------------------------------------
# 9. COAD PCA within common workflow
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COAD WORKFLOW-RESTRICTED PCA")
print("=" * 65)

coad_mask = (
    samples["project"].eq("TCGA-COAD")
    & samples["workflow_version"].eq(
        SHARED_WORKFLOW
    )
)

coad_indices = np.flatnonzero(
    coad_mask.to_numpy()
)

coad_pca, coad_genes, coad_variance = (
    perform_pca(
        "coad_shared_workflow",
        coad_indices
    )
)

plot_pca(
    coad_pca,
    coad_variance,
    "analysis_group",
    "TCGA-COAD — Shared Workflow PCA",
    "tcga_coad_shared_workflow_pca.png"
)


# ------------------------------------------------------------
# 10. READ PCA within common workflow
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("READ WORKFLOW-RESTRICTED PCA")
print("=" * 65)

read_mask = (
    samples["project"].eq("TCGA-READ")
    & samples["workflow_version"].eq(
        SHARED_WORKFLOW
    )
)

read_indices = np.flatnonzero(
    read_mask.to_numpy()
)

read_pca, read_genes, read_variance = (
    perform_pca(
        "read_shared_workflow",
        read_indices
    )
)

plot_pca(
    read_pca,
    read_variance,
    "analysis_group",
    "TCGA-READ — Shared Workflow PCA",
    "tcga_read_shared_workflow_pca.png"
)


# ------------------------------------------------------------
# 11. Examine samples at PCA extremes
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXPRESSION STRUCTURE REVIEW")
print("=" * 65)

# Distance from the global PCA centroid
# is descriptive only.
#
# Samples far from the centroid are NOT
# automatically considered low-quality.

global_pca["pc12_distance"] = np.sqrt(
    global_pca["PC1"] ** 2
    + global_pca["PC2"] ** 2
)

extreme_samples = (
    global_pca
    .sort_values(
        "pc12_distance",
        ascending=False
    )
    .head(12)
)

print("\nSamples farthest from global PCA centroid:")

print(
    extreme_samples[
        [
            "patient_id",
            "sample_id",
            "analysis_group",
            "workflow_short",
            "PC1",
            "PC2",
            "library_size_millions",
            "assignment_percent",
            "n_qc_review_flags"
        ]
    ]
    .round(2)
    .to_string(index=False)
)

print("\nNOTE:")
print(
    "Extreme PCA coordinates do not "
    "automatically indicate poor sample quality."
)


# ------------------------------------------------------------
# 12. Save PCA outputs
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING PCA RESULTS")
print("-" * 65)

pca_tables = {

    "tcga_global_pca_coordinates.csv":
        global_pca,

    "tcga_coad_shared_workflow_pca.csv":
        coad_pca,

    "tcga_read_shared_workflow_pca.csv":
        read_pca,

    "tcga_global_pca_genes.csv":
        global_genes,

    "tcga_coad_pca_genes.csv":
        coad_genes,

    "tcga_read_pca_genes.csv":
        read_genes
}

for filename, dataframe in pca_tables.items():

    dataframe.to_csv(
        RESULTS / filename,
        index=False
    )

print("\nPCA tables saved:", len(pca_tables))


# ------------------------------------------------------------
# 13. Save PCA summary and provenance
# ------------------------------------------------------------

summary_records = []

for name, data, features, variance in [

    (
        "global",
        global_pca,
        global_genes,
        global_variance
    ),

    (
        "coad_shared_workflow",
        coad_pca,
        coad_genes,
        coad_variance
    ),

    (
        "read_shared_workflow",
        read_pca,
        read_genes,
        read_variance
    )

]:

    summary_records.append({

        "analysis": name,

        "samples": len(data),

        "selected_genes": len(features),

        "pc1_variance_percent": variance[0],

        "pc2_variance_percent": variance[1],

        "pc3_variance_percent": variance[2],

        "qc_flagged_samples": int(
            (data["n_qc_review_flags"] > 0).sum()
        )
    })

pca_summary = pd.DataFrame(
    summary_records
)

pca_summary.to_csv(
    RESULTS / "tcga_pca_analysis_summary.csv",
    index=False
)

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.19",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "input_matrix": str(MATRIX_PATH),

    "expression_measure": (
        "Unstranded raw gene counts"
    ),

    "pca_transformation": "log2(CPM + 1)",

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "gene_selection": (
        "Protein-coding genes with CPM >= 1 "
        "in at least 10% of samples, minimum "
        "10 samples, followed by the 2000 "
        "most variable log-CPM genes"
    ),

    "pca_components": 3,

    "random_state": 42,

    "shared_workflow_version":
        SHARED_WORKFLOW,

    "samples_excluded": 0,

    "batch_correction_performed": False,

    "differential_expression_performed": False,

    "limitations": [

        "Log-CPM PCA is exploratory.",

        "Composition-adjusted normalization "
        "and variance-stabilizing transformations "
        "have not yet been evaluated.",

        "PCA does not establish causes "
        "of expression variation.",

        "Workflow version and evidence tier "
        "are strongly confounded.",

        "The shared-workflow subset contains "
        "very few Tier A tumors.",

        "Shared workflow does not eliminate "
        "all potential batch effects.",

        "QC review flags are not exclusions.",

        "Normal tissues are not all paired "
        "with tumor samples."

    ]
}

with open(
    META / "tcga_pca_provenance.json",
    "w"
) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.19 SUMMARY")
print("=" * 65)

print("\nPCA analysis summary:")

print(
    pca_summary
    .round(2)
    .to_string(index=False)
)

print("\nFigures generated: 4")

print("PCA coordinate/gene tables: 6")

print("Summary table: 1")

print("Provenance: SAVED")

print("\nSamples excluded: 0")

print("Batch correction performed: NO")

print("Differential expression performed: NO")

print("\nPATCH 1.19 COMPLETE")

print("=" * 65)